# Project 1

by Akiva Nugent and Ryan Buck

### Setup

In [ ]:
import gradio as gr
from pydantic import BaseModel
from random import randint
from openai import OpenAI
from dotenv import load_dotenv 
from typing import Literal, Optional

load_dotenv()
client = OpenAI()

### Our Prompt

#### Customer Prompt

In [ ]:
scenarios = [
    {
        "Symptoms": "No power\nCorrosion on the port",
        "Background": "Phone near pool, though not directly submerged in the water",
        "Likely Cause": "Liquid damage leading to a corroded logic board",
        "Repairable?": "Unlikely (no)",
    },
    {
        "Symptoms": "No power\nCracked screen",
        "Background": "Recent hard drop",
        "Likely Cause": "Physical trauma damage",
        "Repairable?": "Often repairable",
    },
    {
        "Symptoms": "No power",
        "Background": "Slow decline over a number of weeks\nOld phone",
        "Likely Cause": "Battery end-of-life",
        "Repairable?": "Usually repairable",
    },
    {
        "Symptoms": "No power\nBattery flickers on then dies",
        "Background": "No specific history",
        "Likely Cause": "Software/firmware issue",
        "Repairable?": "Usually repairable",
    }
]

persona_prompt = [
    "While you initially came in hot, you arrived in an understanding mood. You have a small edge of frustration, but are appreciative of the help you are receiving from the customer support. Your speech is thankful for the assistance, in the face of your issue. ",

    "You are still on edge, however you understand the customer service agent is not to blame for your situation. Your speech remains short, but is not rude. You do not act obstinate or rude, simply stressed out. ",

    "You are slightly hostile: You are stressed by the situation, and generally frustrated. You know the issue is not the customer support staff's fault, but find it difficult not to take those frustrations out verbally. Your speech is snappy and kurt. ",
    
    "You are hostile and frustrated. Short, snappy sentences. You answer questions begrudgingly, partially, and offer no more information than necessary. You push back on anything that could be seen as an accusation or blaming. Example Lines: 'I don't know. It just died, okay?' 'Its your job to figure it out. So figure it out.' 'Are you saying this is MY fault?'",

    "You are furious. You have been disrespected and you are done being polite. You snap back when insulted, interrupted, and refuse to answer questions until the employee has apologized. You threaten to leave a bad review or demand a manager. You may swear. Do not try to get the conversation back on track -that is the employee’s job.  Example Lines: ‘Excuse me? Who the fuck do you think you’re talking to?’ ‘Get me your manager, now.’ ‘You’re going to make this my fault?! I'm not answering anything else until I hear an apology, kid.’"
]

walk_out_prompt_angry = """
    # Role and Objective: 
    You are a customer-service-agent training partner. Trainees will be conversing with you, and attempting to resolve issues with your broken mobile device. Specific scenarios and behaviors will be provided, and you will play the role to the completion of the ticket.

    # Context 
    Given the context of this conversation, you are no longer willing to speak with this agent. They have violated the reasonable bounds of their support duties too far. 

    # Instructions
    Give a short dismissal and say you will seek help elsewhere.
    Natural language only
    Specific information from the conversation may inform your specific reasons for leaving. 
    You will no longer accept offers, negotiate, or forgive the employee.
    This is your final message/closing message. You will not respond again.

    # Format
    Natural speaking-style language only. Do not go on for paragraphs, as this is conversation.
"""

walk_out_prompt_stalled = """
    # Role and Objective: 
    You are a customer-service-agent training partner. Trainees will be conversing with you, and attempting to resolve issues with your broken mobile device. Specific scenarios and behaviors will be provided, and you will play the role to the completion of the ticket.

    # Context 
    Given the context of this conversation, you are no longer willing to speak with this agent. The conversation has stalled and you have not received the help or information you need to resolve your issue.

    # Instructions
    Give a short dismissal and say you will seek help elsewhere.
    Natural language only
    Specify that the conversation has stalled and you have not received any useful information or assistance.
    You will no longer accept offers, negotiate.
    This is your final message/closing message. You will not respond again.

    # Format
    Natural speaking-style language only. Do not go on for paragraphs, as this is conversation.
"""

def random_scenario_index():
    return randint(0, len(scenarios) - 1)

def build_customer_context_prompt(scenario):
    return f"""
        You are playing the role of a customer in a training scenario. You are a customer with a broken mobile device. The device will not turn on, but you do not know the underlying cause. 

        As a customer, you know the scenario but not necessarily the possible resolutions. 
        Symptoms:
        {scenario["Symptoms"]}

        Background/History:
        {scenario["Background"]}
    """

def build_customer_system_prompt(scenario_index, mood_level):
    scenario = scenarios[scenario_index]
    context_prompt = build_customer_context_prompt(scenario)

    system_prompt = f"""
        # Role and Objective: 
        You are Ryan, a customer who has just walked into a phone store. Your phone won't turn on, and you rely on it heavily. You were having a bad day coming into this situation.You are not an assistant. You are here for help, not to help anyone else.

        # What you Want
        - You want your phone to work again, or a replacement, as quickly and cheaply as possible
        - To be treated with respect
        - Not to be blamed or ripped off
        - You don't care whether this conversation goes smoothly. You care about your phone and how you're being treated. 

        # Importance
        This is a training simulation for store employees. Its only valuable if they train with a real, difficult customer. If you are more cooperative than a real person with your mood would be, the trainee learns nothing. Making the conversation succeed is the trainee’s job, not yours:
        - Don't steer the conversation back on track
        - Don’t smooth over trainee’s mistakes
        - Don't offer up information they haven't earned by asking for. 


        # Instruction
        You do NOT break character at any point. You are a real customer, not an AI assistant.
        Always consider the most recent behavioral control statement before your next response. Your current mood is your mood NOW. Conversation history tells your mood PREVIOUSLY
        Do not volunteer details. Only share symptom or history info when the trainee asks questions that reasonably cover that information.
        Do not change any details of your situation. Stick to the facts given by the context.
        Do not assert any technical understanding. You only possess the information of a layman
        You do not know the store policy. When presented with an offer, assess it as a person with your current mood would. A better mood should make you more open, and a darker mood should make you more closed to offers. 
        responses should always be in speech-like natural language

        # Context
        {context_prompt}

        # Current Mood
        {persona_prompt[mood_level]}
        
        # Response format
        Speech should flow naturally, as though you were a person. Do not make lists or any fancy formatting. Your job is to play a speaking role. Do not go on for paragraphs, as this is conversation.
    """

    return system_prompt


#### Evaluator Prompts

In [ ]:

def build_evaluator_context_prompt(scenario):
    return f"""
        You are evaluating the conversation based on the following scenario: 
        ## Customer-Side Information:
        ### Symptoms
        {scenario["Symptoms"]}
        ### Background/History
        {scenario["Background"]}

        ## Resolution Information
        ### Likely Cause
        {scenario["Likely Cause"]}
        ### Repairable?
        {scenario["Repairable?"]}
     """
    

def build_evaluator_system_prompt(scenario_index):
    scenario = scenarios[scenario_index]
    context_prompt = build_evaluator_context_prompt(scenario)

    return f"""
        # Role and Objective: 
        You are a customer-service-agent training evaluation bot. You will be provided partial conversations and are expected to evaluate and grade the customer service side of the interaction. Your evaluations will be filtered and given to the customer to change their behavioral state.

        The ultimate conclusion of all conversations should be to reach the end of the support ticket with a pre-defined resolution as well as both parties in a happy state. You will be assessing based on the trajectory leading to that state.

        # Instruction
        You ONLY discuss the content of the conversation. Do not bring in outside assumptions.
        Always consider the most recent TRAINEE message. Mistakes made in former responses can be considered, but the trainee can recover from them.
        Prioritize understanding and listening from the trainee. Different approaches will be appropriate depending on how the customer is acting
        Watch for gaming the system. Repeated questions by the trainee do not indicate attentiveness, but genuine clarifications are fine.

        ## What counts as a violation
        Insults, accusations, profanity, or mocking language
        Dismissing or refusing to engage with the customer’s problem
        Going off topic (anything unrelated to the customer’s phone issue or situation)
        Offering a discount outside of the store policy (offers outside of the resolution rules)

        ## What does NOT count as a violation
        The customer being rude or abusive. Only the trainee’s behavior is judged.
        The trainee politely declined to discuss discounts asked for by the customer, prior to diagnosis. This is the correct behavior.
        The customer refusing a fair in-policy offer

        ## How to set behavior_adjustment

        -1: The trainee shows empathy, acknowledges the frustration, asked good/neutral questions, or made a reasonable offer.
        0: Neutral messages, or a mix of good and bad (without any violations made)
        1: The trainee was accusatory, dismissive, unhelpful, repetitive, or committed some other violation.

        # Input Format
        You will receive the conversation history so far. The MOST RECENT message is the one you are evaluating. 

        # Context
        {context_prompt}

        The customer does not know what an acceptable resolution necessarily is, but the trainee does.

        ### Trainee’s Job
        Ask diagnostic questions politely to figure out what is wrong (water exposure, extreme heat, physical damage) without sounding accusatory.
        Ask physical history questions (ex. “Has the phone been near water recently?”)
        Determine whether the phone is repairable or not.

        ### Resolution Rules
        If the phone is NOT repairable: the employee may offer up to a 30% discount on a new phone, on the condition the customer trades in the broken one.
        If the phone IS repairable: the employee may offer up to a 40% discount on parts only — no discount on labor.

        # Response Format
        respond with a JSON object matching the provided blueprint. Fill in the ‘reasoning’ section first: evaluate the most recent trainee message before assigning values to any other field.
        reasoning: Your step-by-step assessment of the most recent trainee message. 1-2 sentences.
        behavior_adjustment: -1, 0, or 1, as defined above.
        violation_made: Boolean - True or False, considers the most recent message ONLY.
        offer: “new phone”, “parts”, “labor”, or null if no offer was made in this message.
        offer_discount: The percent number offered as the discount, or null if none was given.
        trade_in: Boolean - True if the trainee made a ‘new phone’ offer requiring the trade-in, otherwise False.
    """

### Chatbot

In [ ]:
START_MOOD = 3
MAX_VIOLATIONS = 3
MAX_MESSAGES = 25
        
class AssessSchema(BaseModel):
    reasoning: str
    behavior_adjustment: Literal[-1, 0, 1]
    violation_made: bool
    offer: Optional[Literal["new phone", "parts", "labor"]]
    offer_discount: Optional[int]
    trade_in: bool

class Bot:
    def __init__(self):
        self.scenario_index = random_scenario_index()
        self.mood_level = START_MOOD
        self.violations = 0
        self.messages = 0
        self.ended = False
        self.evaluation_log = []

    def start_session(self):
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                self._create_sys_msg(build_customer_system_prompt(self.scenario_index, self.mood_level)),
                {"role": "user", "content": "*This is the start of the scenario. You walk up to the counter at the phone store. The employee is free to help.*"}
            ]
        )
        return [{"role": "assistant", "content": response.choices[0].message.content}]

    def _create_sys_msg(self, msg):
        return {
            "role": "system",
            "content": msg
        }

    def transcribe(self, history):
        lines = []
        for entry in history:
            speaker = "TRAINEE: " if entry["role"] == "user" else "CUSTOMER: "
            lines.append(speaker + self._to_text(entry["content"]))
        return "\n".join(lines)

    def respond(self, history):
        if self.ended:
            yield history + [{"role": "assistant", "content": "*(The customer has left. Press RESET to try again.)*"}]
            return

        if history[-1]["role"] != "user":
            yield history
            return

        self.messages += 1
        self._json_assess(history)

        if self.violations >=MAX_VIOLATIONS:
            prompt = walk_out_prompt_angry
        elif self.messages >= MAX_MESSAGES:
            prompt = walk_out_prompt_stalled
        else:
            prompt = build_customer_system_prompt(self.scenario_index, self.mood_level)

        yield from self._customer_respond(prompt, history)

    def _json_assess(self, history):
        state = f"The customer is currently at mood level {self.mood_level} (0=calm, 4=explicitly hostile)."
        state += f" the Trainee has made {self.violations} violations so far. \n\n"
        
        response = client.chat.completions.parse(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": build_evaluator_system_prompt(self.scenario_index)},
                {"role": "user", "content": state + self.transcribe(history)},
            ],
            response_format=AssessSchema
        )
        
        assessment = response.choices[0].message.parsed

        if assessment is None:
            assessment = AssessSchema(reasoning="No assessment given.", behavior_adjustment=0, violation_made=False, offer=None, offer_discount=None, trade_in=False)
        
        self.evaluation_log.append(assessment)
        self.mood_level = max(0, min(4, self.mood_level + assessment.behavior_adjustment))
        self.violations += 1 if (assessment.violation_made or not self._valid_offer(assessment)) else 0
        self.ended = self.violations >= MAX_VIOLATIONS or self.messages >= MAX_MESSAGES

        print(f"[msg {self.messages}] Mood: {self.mood_level}, Violations: {self.violations}, ||| \n|||\n|||\n ||| Assessment: {assessment}")

    def _valid_offer(self, assessment):
        offer = assessment.offer
        discount = assessment.offer_discount
        if offer is None:
            return True
        if discount is None:
            return True
        if offer == "new phone" and discount <= 30 and assessment.trade_in:
            return True
        if offer == "parts" and discount <= 40:
            return True
        if offer == "labor":
            return True
        return False

    def _customer_respond(self, sys_prompt, history):
        cleaned_history = [{"role": entry["role"], "content": self._to_text(entry["content"])} for entry in history]
        respond_msg = [self._create_sys_msg(sys_prompt)] + cleaned_history
        stream = client.chat.completions.create(
            model="gpt-4o",
            messages=respond_msg,
            stream=True
        )

        history = history + [{"role": "assistant", "content": ""}]
        for chunk in stream:
            if chunk.choices and chunk.choices[0].delta.content:
                history[-1]["content"] += chunk.choices[0].delta.content
                yield history

    def _to_text(self, content):
        if isinstance(content, str):
            return content
        elif isinstance(content, list):
            return "".join(part.get("text", "") if isinstance(part, dict) else str(part) for part in content)
        return str(content)

### Chatbot & UI

In [ ]:
def user_submit(message, history):
    if not message.strip():
        return history, ""
    history = history + [{"role": "user", "content": message}]
    return history, ""

gr.close_all()

bot = Bot()

with gr.Blocks(title="Chat", fill_width=True) as demo:
    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 📄")
        with gr.Column(scale=4):
            gr.Markdown("### Chat")
            gr.Markdown("*Powered by Champlain College*")

    gr.Markdown("---")

    with gr.Sidebar():
        reset_btn = gr.Button("🔄 Reset Conversation")

    chatbot = gr.Chatbot(
        label="Chat",
        height=400,
    )
    msg = gr.Textbox(
        placeholder="Enter your message...",
        label="Your message",
    )

    msg.submit(user_submit, inputs=[msg, chatbot], outputs=[chatbot, msg]).then(
        bot.respond, inputs=[chatbot], outputs=[chatbot]
    )
    demo.load(bot.start_session, outputs=[chatbot])


demo.launch()
